[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sibbirhossain/finguard-xai/blob/main/notebooks/07_public_benchmark_ieee_cis.ipynb)

# 07 · Public Benchmark: IEEE-CIS Fraud Detection

Runs the identical pipeline on the public **IEEE-CIS Fraud Detection** dataset: about 590,000 real, anonymized e-commerce transactions provided by Vesta Corporation via Kaggle.

**You must accept the competition rules on Kaggle** and supply your own Kaggle API token. The data cannot be redistributed, so it is never stored in this repository.

Entity mapping, documented in `src/data/ieee_cis.py`:
- **card** = card1–card5 + addr1
- **device** = DeviceInfo
- **merchant** = ProductCD + addr2
- **network identity** = purchaser e-mail domain

The dataset publishes no merchant IDs or IPs, so these are proxies.

In [ ]:
# --- Setup: works in Google Colab and inside a local clone ---
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
if os.path.exists("../src/models"):
    os.chdir("..")
elif not os.path.exists("src/models"):
    if not os.path.exists("finguard-xai"):
        subprocess.run(["git", "clone", "-q", "https://github.com/sibbirhossain/finguard-xai.git"], check=True)
    os.chdir("finguard-xai")
try:
    import torch_geometric  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)
sys.path.insert(0, os.getcwd())
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("ready:", os.getcwd())

In [ ]:
# 1) Upload kaggle.json (Kaggle -> Settings -> API -> Create New Token), then download the data
from pathlib import Path
if not Path("data/ieee-cis/train_transaction.csv").exists():
    try:
        from google.colab import files  # noqa: F401  (Colab only)
        files.upload()                  # choose kaggle.json
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    except ImportError:
        print("Not in Colab: place kaggle.json in ~/.kaggle/ first.")
    subprocess.run(["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", "data/ieee-cis"], check=True)
    subprocess.run(["unzip", "-q", "-o", "data/ieee-cis/ieee-fraud-detection.zip", "-d", "data/ieee-cis"], check=True)
print(sorted(p.name for p in Path("data/ieee-cis").glob("*.csv")))

In [ ]:
# 2) Train + evaluate (chronological split). Start with 150k rows; remove --max-rows for the full set (GPU recommended).
from src.models.train import main as train_main
report = train_main(["--data", "ieee-cis", "--data-dir", "data/ieee-cis", "--max-rows", "150000", "--epochs", "4",
                     "--out", "artifacts/ieee", "--report", "reports/ieee-cis/RESULTS.md", "--bench-n", "300"])

After running, commit `reports/ieee-cis/RESULTS.md`. Those numbers will be **observed on public data** and can be compared directly with published IEEE-CIS results. For robust claims, repeat with several `--seed` values.